# 환경 설정

## 기본 설정

In [ ]:
# GPU 상태 및 VRAM 확인
!nvidia-smi

In [ ]:
from google.colab import drive

# Google Drive를 마운트하고 작업 디렉토리를 Drive 내 고정 경로로 설정한다.
# Drive를 사용하면 세션이 끊겨도 모델 캐시·키·DB가 유지된다.
drive.mount('/content/drive')

import os
work_dir = '/content/drive/MyDrive/ckks_friendly_rag'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"작업 디렉토리: {os.getcwd()}")

## 패키지 설치

In [ ]:
# RAG 파이프라인 의존성
!pip install -q sentence-transformers chromadb langchain langchain-community langchain_chroma
!pip install -q pypdf python-docx tqdm numpy pandas

# Desilo FHE 라이브러리 (CKKS 암호화 엔진)
!pip install -q desilofhe

## 라이브러리 임포트

In [ ]:
from pathlib import Path
import os
import sqlite3

from langchain_chroma import Chroma

import desilofhe
from desilofhe import Engine

import numpy as np

### 경로 변수

In [ ]:
base_path = Path(work_dir)
nomic_dir = base_path / 'PoC_nomic-embed-v1.5 (1)'

# FHE 키 저장 디렉토리 및 파일 경로
KEY_PATH        = base_path / 'keys'
SECRET_KEY_PATH = KEY_PATH / 'secret.key'
PUBLIC_KEY_PATH = KEY_PATH / 'public.key'
REL_KEY_PATH    = KEY_PATH / 'relin.key'
ROT_KEY_PATH    = KEY_PATH / 'rotation.key'

# 암호화 벡터 DB (행 기준: 문서당 1 ciphertext)
db_path     = nomic_dir / 'encrypted.db'
# 암호화 벡터 DB (열 기준: 차원당 1 ciphertext, 행렬곱 Retrieval에 사용)
col_db_path = base_path / 'new_t_row_encrypted.db'
# ChromaDB 평문 벡터 스토어 경로
chroma_path = base_path / 'chroma.sqlite3'

# 입력 문서
doc_path = base_path / 'cat-facts.txt'

# 모델 캐시 (Drive에 저장하면 세션 재시작 시 재다운로드 불필요)
cache_dir = '/content/drive/MyDrive/RAG_Project/model_cache'

### 임베딩 모델

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

os.makedirs(cache_dir, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# nomic-embed-text-v1.5: Matryoshka 표현 학습 지원 모델.
# truncate_dim=256으로 차원을 축소해 FHE slot_count(256)에 맞춤.
# 대안 모델: nomic-ai/nomic-embed-text-v2-moe (더 높은 품질, 더 큰 차원)
model = SentenceTransformer(
    'nomic-ai/nomic-embed-text-v1.5',
    trust_remote_code=True,
    device=device,
    cache_folder=cache_dir
)

print(f'모델 로드 완료 ({device})')
print(f'원래 차원: {model.get_sentence_embedding_dimension()}')

### FHE 엔진 및 하이퍼파라미터

In [ ]:
# CKKS FHE 엔진 초기화.
# slot_count: 한 ciphertext에 패킹할 수 있는 부동소수점 값의 수.
# dim과 반드시 일치해야 하며, 한 번 DB를 구축하면 변경 불가.
engine = Engine(slot_count=256, mode='gpu')

model_name = 'nomic/nomic-embed-text-v1.5'
slot_count = engine.slot_count   # 256
dim        = 256                  # 사용할 임베딩 차원 (slot_count와 동일하게 설정)

# Top-K는 반드시 2의 거듭제곱이어야 한다.
# rotate-add 트리에서 2^ceil(log2(k)) 번의 rotation이 필요하기 때문.
TOP_K_SIZE = 4

print(f'slot_count: {slot_count}, dim: {dim}, TOP_K_SIZE: {TOP_K_SIZE}')

## 유틸리티 함수

### SQLite 저장

In [ ]:
def save_ciphertexts_sqlite(db_path: str, ids, cipher_bytes_list, table='encrypted_vectors'):
    """
    직렬화된 ciphertext bytes를 SQLite에 BLOB으로 저장한다.

    Args:
        db_path:           SQLite 파일 경로
        ids:               각 ciphertext의 문자열 ID 리스트
        cipher_bytes_list: engine.serialize_ciphertext()의 결과 bytes 리스트
        table:             저장할 테이블명 (기본값: 'encrypted_vectors')
    """
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id TEXT PRIMARY KEY,
            ciphertext BLOB
        )
    """)

    # executemany는 루프 INSERT보다 빠름
    cur.executemany(
        f"INSERT OR REPLACE INTO {table} (id, ciphertext) VALUES (?, ?)",
        [(doc_id, sqlite3.Binary(blob)) for doc_id, blob in zip(ids, cipher_bytes_list)]
    )

    conn.commit()
    conn.close()

### SQLite 로드

In [ ]:
def load_ciphertexts_sqlite(db_path: str, engine, table='encrypted_vectors'):
    """
    SQLite에서 ciphertext BLOB을 읽어 역직렬화된 ciphertext 객체 리스트로 반환한다.

    주의: ID가 TEXT로 저장되어 있으므로 반드시 CAST(id AS INTEGER)로 정렬해야
    문서 인덱스와 점수 배열 간 순서 불일치를 방지할 수 있다.
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur  = conn.cursor()

    cur.execute(f"""
        SELECT id, ciphertext
        FROM {table}
        ORDER BY CAST(id AS INTEGER)
    """)

    out = []
    for row in cur.fetchall():
        ct = engine.deserialize_ciphertext(row['ciphertext'])
        out.append(ct)

    conn.close()
    return out

### 벡터 암호화

In [ ]:
def encrypt_values(vector_list, engine, public_key):
    """
    정규화된 임베딩 벡터 리스트를 CKKS로 암호화하고 직렬화된 bytes를 반환한다.

    각 벡터를 slot_count 크기의 배열로 패딩(zero-padding)한 뒤 암호화한다.
    dim < slot_count인 경우 나머지 슬롯은 0으로 채워 내적 연산에 영향을 주지 않는다.
    """
    encrypted_embeddings = []
    slot_count = engine.slot_count

    for vec in vector_list:
        packed = [0] * slot_count
        packed[0:dim] = vec[0:dim]            # 유효 차원만 앞 슬롯에 배치
        ctx = engine.encrypt(packed, public_key)
        cipher_bytes = engine.serialize_ciphertext(ctx)
        encrypted_embeddings.append(cipher_bytes)

    return encrypted_embeddings

### L2 정규화

In [ ]:
def l2_normalize(vec, eps=1e-12):
    """
    벡터를 L2 정규화하여 반환한다.

    정규화된 벡터끼리의 내적은 코사인 유사도와 동일하므로,
    암호화된 내적만으로 의미 기반 유사도 검색이 가능해진다.
    eps는 영벡터 분모 0 방지용이다.
    """
    vec  = np.asarray(vec, dtype=np.float64)
    norm = np.linalg.norm(vec)
    return vec / (norm + eps)

### 암호화 내적 (Encrypted Inner Product)

In [ ]:
def encrypted_inner_product(ct_query, ct_db, engine, rel_key, rot_key):
    """
    두 암호화된 벡터의 내적을 FHE 상태에서 계산한다.

    알고리즘:
        1. 원소별 곱셈: ct_mul[i] = ct_query[i] * ct_db[i]
           (CKKS 곱셈 후 ciphertext degree가 올라가므로 rel_key로 재선형화)
        2. rotate-and-add 트리: 슬롯 합산
           step=1,2,4,...,dim2 동안 ct_mul을 step만큼 회전시켜 더한다.
           결과로 모든 슬롯에 내적값 S가 복제된다.
    """
    ct_mul = engine.multiply(ct_query, ct_db, rel_key)

    # dim을 올림 2의 거듭제곱으로 맞춰 rotate-add 트리 깊이를 최소화
    dim2 = 1 << (dim - 1).bit_length()
    step = 1
    while step < dim2:
        ct_rot = engine.rotate(ct_mul, rot_key, step)
        ct_mul = engine.add(ct_mul, ct_rot)
        step <<= 1

    return ct_mul  # 모든 슬롯에 내적값이 복제된 상태

### Top-K 선택

In [ ]:
def top_k_retrieval(results, k=10):
    """
    점수 기준 내림차순으로 정렬하여 상위 k개를 반환한다.
    results: [{'id': int, 'score': float}, ...]
    """
    sorted_results = sorted(results, key=lambda x: x['score'], reverse=True)
    return sorted_results[:k]

### 인덱스 재암호화 (향후 프라이버시 보호용)

In [ ]:
def encrypt_indices(results, engine, public_key):
    """
    Top-K 결과의 문서 인덱스를 암호화한다.
    현재는 서버 측에서 인덱스를 평문으로 처리하지만,
    완전한 프라이버시 보호 파이프라인에서는 이 함수를 통해
    어떤 문서가 선택됐는지 서버에게 노출되지 않도록 한다.
    """
    encrypted_indices = []
    for res in results:
        encrypted_indices.append(engine.encrypt_ciphertext(res['id'], public_key))
    return encrypted_indices

### 암호화 행렬곱 (Encrypted Matrix Multiplication)

In [ ]:
def sum_first_n_slots(ct, n, engine, rot_key):
    """
    Enc([x0, x1, ..., x_{n-1}, 0, 0, ...]) → Enc([S, S, S, ...])
    (S = sum_{i=0}^{n-1} x_i)

    rotate-add 트리로 앞 n개 슬롯의 합을 모든 슬롯에 복제한다.
    """
    dim2 = 1 << (n - 1).bit_length()  # n 이상의 최소 2의 거듭제곱
    acc  = ct
    step = 1
    while step < dim2:
        acc  = engine.add(acc, engine.rotate(acc, rot_key, step))
        step <<= 1
    return acc


def encrypted_dot(ct_a, ct_b, n, engine, rel_key, rot_key):
    """
    패킹된 암호화 벡터 ct_a, ct_b의 내적을 계산한다 (유효 길이 n).
    결과 ciphertext의 모든 슬롯에 동일한 내적값이 복제된다.
    """
    ct_mul = engine.multiply(ct_a, ct_b, rel_key)
    return sum_first_n_slots(ct_mul, n, engine, rot_key)


def encrypted_matmul(A_rows, B_cols, n, engine, rel_key, rot_key):
    """
    암호화된 행렬 A (k×n)와 B (n×m)의 곱 C (k×m)를 계산한다.
    A_rows: 암호화된 행 벡터 리스트 (len=k)
    B_cols: 암호화된 열 벡터 리스트 (len=m)
    C[r][c] = encrypted_dot(A[r, :], B[:, c])

    Retrieval 맥락에서:
      A = 선택 행렬 (Selector, k×doc_count)
      B = 전치된 임베딩 DB (doc_count×dim → 열 기준 암호화)
    → C[r] = r번째 Top-K 문서의 임베딩 벡터 (암호화 상태)
    """
    k = len(A_rows)
    m = len(B_cols)

    C = [[None] * m for _ in range(k)]
    for r in range(k):
        for c in range(m):
            C[r][c] = encrypted_dot(A_rows[r], B_cols[c], n, engine, rel_key, rot_key)
    return C

### Selector 벡터

In [ ]:
def create_selector_vector_row(i, n):
    """i번째 원소만 1인 길이 n의 one-hot 행 벡터를 반환한다."""
    vec    = np.zeros(n, dtype=np.float64)
    vec[i] = 1.0
    return vec


def rows_to_cols(B_rows):
    """행 기준 리스트를 열 기준 리스트로 전치한다.
    B_rows: [[dim], [dim], ...]  (D개)
    return: [[D],   [D],   ...]  (dim개)
    """
    return [list(col) for col in zip(*B_rows)]


def create_selector_vector_pt(target_indices, n):
    """
    target_indices에 해당하는 평문 one-hot 행 벡터 리스트를 반환한다.
    행렬곱 시 평문 selector로 사용된다 (encrypted_matmul의 A_rows 인자).
    """
    return [create_selector_vector_row(i, n) for i in target_indices]


def create_selector_vector(target_indices, n):
    """
    target_indices에 해당하는 암호화된 one-hot 행 벡터 리스트를 반환한다.
    서버가 어떤 문서 인덱스를 선택했는지 알 수 없도록 selector를 암호화한다.
    """
    enc_selectors = []
    for i in target_indices:
        row    = create_selector_vector_row(i, n)
        ct_row = engine.encrypt(row, public_key)
        enc_selectors.append(ct_row)
    return enc_selectors

## 키 관리

CKKS에서는 4가지 키가 필요하다:
- **Secret key**: 복호화 전용. 클라이언트만 보유.
- **Public key**: 암호화 전용. 서버와 공유 가능.
- **Relinearization key**: 두 ciphertext를 곱한 뒤 degree를 낮추는 데 사용.
- **Rotation key**: 슬롯 회전(rotate-and-add)에 사용.

키 파일이 이미 존재하면 로드하고, 없으면 생성 후 Drive에 저장한다.
키를 새로 생성하면 기존 암호화 DB를 모두 재구축해야 한다.

In [ ]:
os.makedirs(KEY_PATH, exist_ok=True)

# Secret key: 복호화 및 다른 키 유도에 사용
if SECRET_KEY_PATH.exists():
    with open(SECRET_KEY_PATH, 'rb') as f:
        secret_key = engine.deserialize_secret_key(f.read())
else:
    secret_key   = engine.create_secret_key()
    secret_bytes = engine.serialize_secret_key(secret_key)
    with open(SECRET_KEY_PATH, 'wb') as f:
        f.write(secret_bytes)

# Public key: 임베딩 암호화에 사용 (서버 공유 가능)
if PUBLIC_KEY_PATH.exists():
    with open(PUBLIC_KEY_PATH, 'rb') as f:
        public_key = engine.deserialize_public_key(f.read())
else:
    public_key   = engine.create_public_key(secret_key)
    public_bytes = engine.serialize_public_key(public_key)
    with open(PUBLIC_KEY_PATH, 'wb') as f:
        f.write(public_bytes)

print(f'Engine slot count: {engine.slot_count}')
print('Secret / Public key 로드 완료')

In [ ]:
# Relinearization key: 암호문 곱셈 후 degree 감소에 사용
if REL_KEY_PATH.exists():
    with open(REL_KEY_PATH, 'rb') as f:
        rel_key = engine.deserialize_relinearization_key(f.read())
else:
    rel_key   = engine.create_relinearization_key(secret_key)
    rel_bytes = engine.serialize_relinearization_key(rel_key)
    with open(REL_KEY_PATH, 'wb') as f:
        f.write(rel_bytes)

# Rotation key: rotate-and-add 슬롯 합산에 사용
if ROT_KEY_PATH.exists():
    with open(ROT_KEY_PATH, 'rb') as f:
        rot_key = engine.deserialize_rotation_key(f.read())
else:
    rot_key   = engine.create_rotation_key(secret_key)
    rot_bytes = engine.serialize_rotation_key(rot_key)
    with open(ROT_KEY_PATH, 'wb') as f:
        f.write(rot_bytes)

print(f'Relinearization key: {rel_key}')
print(f'Rotation key: {rot_key}')

# DB 구축

### 문서 로드 및 청킹

In [ ]:
from langchain_community.document_loaders import TextLoader

# LangChain Document 형식으로 원본 파일 로드
loader  = TextLoader(doc_path)
dataset = loader.load()
print(f'원본 문서 수: {len(dataset)}')

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size=600: 한 청크의 최대 문자 수.
# chunk_overlap=0: 청크 간 중복 없음 (암호화 DB 인덱스와 1:1 대응 유지).
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=0
)

docs = text_splitter.split_documents(dataset)
print(f'청킹 완료: {len(docs)}개 청크')

### 임베딩 → ChromaDB 저장

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# SentenceTransformer 모델을 LangChain 임베딩 인터페이스로 래핑.
# ChromaDB에 평문 임베딩을 저장해 두면, 이후 정규화·암호화 작업의
# 원본 벡터로 활용하거나 디버그 시 평문 검색을 비교에 쓸 수 있다.
langchain_embeddings = HuggingFaceEmbeddings(
    model_name='nomic-ai/nomic-embed-text-v1.5',
    model_kwargs={'device': device, 'trust_remote_code': True},
    show_progress=False
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=langchain_embeddings,
    persist_directory=chroma_path,
    collection_name='nomic_embed'
)

# 이후 처리(정규화·암호화)를 위해 임베딩과 원문을 메모리에 로드
vec_db = vectorstore.get(include=['embeddings', 'documents'])

print(f'저장된 문서 수: {len(vec_db["ids"])}')
print(f'임베딩 차원: {len(vec_db["embeddings"][0])}')

### 정규화

내적이 코사인 유사도와 동일해지도록 임베딩을 L2 정규화한다.

In [ ]:
normalized_embeddings = [
    l2_normalize(emb) for emb in vec_db['embeddings']
]
print(f'정규화 완료: {len(normalized_embeddings)}개')

### 행 기준 암호화 (Row DB)

각 문서 임베딩을 독립적으로 암호화하여 저장한다.  
쿼리 ciphertext와의 내적 계산(Retrieval 2단계)에 사용한다.

In [ ]:
encrypted_embeddings = []

for emb in normalized_embeddings:
    # slot_count 크기 배열에 zero-padding 후 앞 dim 슬롯에 임베딩 배치
    packed = [0.0] * slot_count
    packed[0:dim] = emb[0:dim]
    ctx          = engine.encrypt(packed, public_key)
    cipher_bytes = engine.serialize_ciphertext(ctx)
    encrypted_embeddings.append(cipher_bytes)

print(f'암호화 완료: {len(encrypted_embeddings)}개')

#### 암호화된 데이터 저장 (SQLite)

In [ ]:
# 정수 인덱스를 문자열로 변환 (SQLite ID는 TEXT 타입)
# 로드 시 ORDER BY CAST(id AS INTEGER)로 반드시 숫자 정렬해야 함
index_arr = [str(i) for i in range(len(encrypted_embeddings))]

os.makedirs(db_path.parent, exist_ok=True)
save_ciphertexts_sqlite(db_path, index_arr, encrypted_embeddings)
print(f'Row DB 저장 완료: {db_path}')

### 열 기준 암호화 (Column DB)

행렬을 **전치**하여 차원 축(열) 단위로 암호화한다.  
원본 행렬 shape: `(n_docs, slot_count)` → 전치 후: `(slot_count, n_docs)`  
전치된 각 행(=원본의 각 열)을 하나의 ciphertext로 암호화하면  
`slot_count`개의 ciphertext가 생성된다.  

Retrieval 4단계에서 `Selector(k×n_docs) × EmbDB(n_docs×dim)` 행렬곱을 수행할 때,  
이 열 기준 DB를 B의 열 벡터로 사용한다.

In [ ]:
import chromadb
import chromadb.utils.embedding_functions as embedding_functions

# 로컬 ChromaDB 클라이언트로 평문 임베딩 수집
chroma_client = chromadb.PersistentClient(path=r'.')

col_embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='nomic-ai/nomic-embed-text-v1.5',
    device='cpu',
    normalize_embeddings=True,   # L2 정규화 포함
    trust_remote_code=True,
    truncate_dim=256              # dim=256으로 축소
)

# 컬렉션 초기화 (기존 컬렉션이 있으면 삭제 후 재생성)
try:
    chroma_client.delete_collection('nomic_v1.5_T')
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name='nomic_v1.5_T',
    embedding_function=col_embed_fn
)

# 문서 청킹 및 ChromaDB 등록 (chunk_overlap=0으로 행렬 인덱스와 1:1 대응)
with open(base_path / 'cat-facts.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

col_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=['\n\n', '\n', '. ']
)
ptxt_list = col_splitter.split_text(full_text)
index_arr = [f'id{i+1}' for i in range(len(ptxt_list))]

collection.add(ids=index_arr, documents=ptxt_list)

# 평문 임베딩 수집 후 (n_docs × slot_count) 행렬로 패딩
db_collection   = collection.get(ids=index_arr, include=['embeddings'])
embeddings_list = db_collection['embeddings']
num_items       = len(embeddings_list)
SLOT_COUNT      = engine.slot_count

padded_matrix = np.zeros((num_items, SLOT_COUNT))
for i, arr in enumerate(embeddings_list):
    limit = min(len(arr), SLOT_COUNT)
    padded_matrix[i, :limit] = arr[:limit]

# 전치: (n_docs × SLOT_COUNT) → (SLOT_COUNT × n_docs)
# 전치 행렬의 j번째 행 = 모든 문서의 j번째 차원값 벡터
transposed_matrix = padded_matrix.T

print(f'문서 수: {num_items}, padded_matrix: {padded_matrix.shape}')
print(f'transposed_matrix: {transposed_matrix.shape}')

In [ ]:
# ── 열 기준 암호화 DB 저장 ───────────────────────────────────────────
# 각 열(j번째 차원)을 하나의 ciphertext로 암호화.
# ciphertext j의 슬롯 0..num_items-1 = 모든 문서의 j번째 차원값.
t_db_path = col_db_path
if os.path.exists(t_db_path):
    os.remove(t_db_path)

conn   = sqlite3.connect(t_db_path)
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS encrypted_vectors (
        id TEXT PRIMARY KEY,
        ciphertext BLOB
    )
""")

for j in range(SLOT_COUNT):
    col_vec = padded_matrix[:, j]          # j번째 열: 모든 문서의 j차원 값
    packed_col = np.zeros(SLOT_COUNT, dtype=np.float64)
    packed_col[0:num_items] = col_vec      # num_items 이후는 zero-pad
    ct   = engine.encrypt(packed_col, public_key)
    blob = engine.serialize_ciphertext(ct)
    cursor.execute(
        'INSERT INTO encrypted_vectors (id, ciphertext) VALUES (?, ?)',
        (str(j), sqlite3.Binary(blob))
    )

conn.commit()

# ── 저장 검증 ──────────────────────────────────────────────────────────
print('열 기준 암호화 DB 검증 중...')
for j in range(SLOT_COUNT):
    original_col = padded_matrix[:, j]
    cursor.execute('SELECT ciphertext FROM encrypted_vectors WHERE id = ?', (str(j),))
    row = cursor.fetchone()
    if row:
        loaded_ct  = engine.deserialize_ciphertext(row[0])
        decrypted  = engine.decrypt(loaded_ct, secret_key)
        is_same    = np.allclose(original_col, decrypted[:num_items], atol=1e-5, rtol=1e-8)
        status     = 'OK' if is_same else 'FAIL'
        if not is_same:
            print(f'  [FAIL] column {j}: 벡터 불일치')
    else:
        print(f'  [FAIL] column {j}: DB에서 찾을 수 없음')

conn.close()
print(f'열 기준 DB 저장 완료: {t_db_path}')

In [ ]:
# ── 행 기준 암호화 DB 저장 및 검증 (helper 함수) ────────────────────────
# padded_matrix (n_docs × SLOT_COUNT)의 각 행을 하나의 ciphertext로 암호화.
# 쿼리 ciphertext와의 encrypted_inner_product 계산에 사용.
def db_save_verify(matrix, path):
    if os.path.exists(path):
        os.remove(path)

    conn   = sqlite3.connect(path)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS encrypted_vectors (
            id TEXT PRIMARY KEY,
            ciphertext BLOB
        )
    """)

    for i, row in enumerate(matrix):
        # 행 길이가 SLOT_COUNT보다 짧으면 zero-padding
        if len(row) < SLOT_COUNT:
            padded_row = np.zeros(SLOT_COUNT, dtype=np.float64)
            padded_row[0:len(row)] = row
            vector_to_encrypt = padded_row
        else:
            vector_to_encrypt = row

        ct   = engine.encrypt(vector_to_encrypt, public_key)
        blob = engine.serialize_ciphertext(ct)
        cursor.execute(
            'INSERT INTO encrypted_vectors (id, ciphertext) VALUES (?, ?)',
            (str(i), blob)
        )

    conn.commit()

    # 저장 검증
    print(f'검증 중: {path}')
    for i in range(len(matrix)):
        original_row = matrix[i]
        cursor.execute('SELECT ciphertext FROM encrypted_vectors WHERE id = ?', (str(i),))
        row_from_db = cursor.fetchone()
        if row_from_db:
            loaded_ct = engine.deserialize_ciphertext(row_from_db[0])
            decrypted = engine.decrypt(loaded_ct, secret_key)
            valid_len = len(original_row)
            is_same   = np.allclose(original_row, decrypted[:valid_len], atol=1e-5, rtol=1e-8)
            if not is_same:
                print(f'  [FAIL] row {i}: 벡터 불일치')
        else:
            print(f'  [FAIL] row {i}: DB에서 찾을 수 없음')

    conn.close()
    print('검증 완료')


# 행 기준 Row DB 저장
print('행 기준 암호화 DB 저장 중...')
db_save_verify(padded_matrix, str(db_path))

# 전치 행렬의 행 기준 저장 (= 원본의 열 기준 암호화, 정규화 포함)
# 각 행의 L2 노름으로 나누어 정규화
norm = np.linalg.norm(transposed_matrix, axis=0)
print('\n전치+정규화된 행 기준 암호화 DB 저장 중...')
db_save_verify(transposed_matrix / norm, str(col_db_path))

# DB 로드

#### 평문 임베딩 (ChromaDB)

In [ ]:
# ChromaDB에서 평문 임베딩 로드 (검증 및 비교용)
persist_db = Chroma(
    persist_directory=db_path,
    embedding_function=col_embed_fn,
    collection_name='nomic_embed'
)
vec_db = persist_db.get(include=['embeddings', 'documents'])

#### 암호화된 벡터 (SQLite)

1. SQLite에서 ciphertext BLOB을 bytes로 꺼내고  
2. `engine.deserialize_ciphertext(bytes)`로 ciphertext 객체로 복원한다.  
3. `ORDER BY CAST(id AS INTEGER)` 정렬이 필수 — TEXT 정렬 시 '10' < '9'가 되어 순서가 어긋난다.

In [ ]:
# 테이블명 확인 (DB 구조 검사)
con = sqlite3.connect(col_db_path)
cur = con.cursor()
table_name_result = cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchone()
con.close()

if table_name_result is not None:
    table_name = table_name_result[0]
    print(f'테이블명: {table_name}')
else:
    table_name = 'encrypted_vectors'
    print(f'Warning: 테이블 없음. 기본값 사용: {table_name}')

In [ ]:
def load_ciphertexts_ordered(db_path: str, engine, table='encrypted_vectors'):
    """
    SQLite에서 ciphertext를 ID 숫자 오름차순으로 로드한다.
    CAST(id AS INTEGER) 정렬로 TEXT '10' > '9' 문제를 방지한다.
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur  = conn.cursor()

    cur.execute(f"""
        SELECT id, ciphertext
        FROM {table}
        ORDER BY CAST(id AS INTEGER)
    """)

    out = []
    for row in cur.fetchall():
        ct = engine.deserialize_ciphertext(row['ciphertext'])
        out.append(ct)

    conn.close()
    return out

In [ ]:
# 열 기준 암호화 DB 로드 (행렬곱 Retrieval용)
# t_encrypted_db_cts[j] = j번째 차원에 대한 모든 문서 값이 패킹된 ciphertext
t_encrypted_db_cts = load_ciphertexts_ordered(col_db_path, engine)
print(f'열 기준 DB 로드: {len(t_encrypted_db_cts)}개 ciphertext')

In [ ]:
# 행 기준 암호화 DB 로드 (내적 기반 유사도 검색용)
# encrypted_db_cts[i] = i번째 문서 임베딩 전체가 패킹된 ciphertext
encrypted_db_cts = load_ciphertexts_sqlite(db_path, engine, 'encrypted_vectors')
print(f'행 기준 DB 로드: {len(encrypted_db_cts)}개 ciphertext')

# Retrieval

프라이버시 보호 RAG Retrieval 4단계:
1. **쿼리 암호화**: 클라이언트가 쿼리를 임베딩 후 암호화하여 서버로 전송
2. **암호화된 내적**: 서버가 쿼리 ciphertext와 DB ciphertext 간 내적을 FHE로 계산
3. **점수 복호화 + Top-K**: 클라이언트가 복호화 후 상위 k개 선택 *(인하우스)*
4. **암호화된 문서 복원**: 선택된 문서를 암호화 상태로 행렬곱으로 추출

### 1단계: 쿼리 암호화

In [ ]:
# query = input('질문을 입력하세요: ')  # 대화형 실행 시 활성화
query = 'what is bezoar?'

# 쿼리를 임베딩하고 L2 정규화 (normalize_embeddings=True)
# truncate_dim=256으로 DB와 동일한 차원 사용
emb_query = model.encode(query, normalize_embeddings=True, truncate_dim=256)

# 쿼리 임베딩을 CKKS로 암호화 — 이후 서버는 쿼리 내용을 알 수 없음
ct_q = engine.encrypt(emb_query, public_key)

### 2단계: 암호화된 내적 계산

서버가 암호화된 쿼리와 DB의 모든 문서 간 내적을 FHE 상태에서 계산한다.  
결과 ciphertext는 서버가 복호화할 수 없다.

In [ ]:
inner_products = []

# 선형 스캔: 쿼리 ciphertext와 DB 각 문서 ciphertext 간 내적 계산
# encrypted_inner_product: 원소곱 + rotate-and-add 슬롯 합산
for ct_db in encrypted_db_cts:
    ip = encrypted_inner_product(ct_q, ct_db, engine, rel_key, rot_key)
    inner_products.append(ip)

### 3단계: 점수 복호화 및 Top-K 선택 (인하우스)

> **인하우스(In-house)**: 이 단계는 클라이언트 측에서 복호화하여 처리한다.  
> 완전한 프라이버시 보호를 위해서는 암호화된 상태에서 정렬이 필요하지만,  
> 현재 PoC에서는 클라이언트가 점수를 복호화한 뒤 상위 k개를 선택한다.

In [ ]:
# 각 내적 ciphertext의 슬롯 0번 값이 코사인 유사도 점수
indices      = list(range(len(inner_products)))
result_scores = []

for i, ip in enumerate(inner_products):
    decrypted_score = engine.decrypt(ip, secret_key)[0]  # 슬롯 0 = 내적값
    result_scores.append({'id': indices[i], 'score': decrypted_score})

top_k_results = top_k_retrieval(result_scores, k=TOP_K_SIZE)
for res in top_k_results:
    print(f"ID: {res['id']}, Score: {res['score']:.6f}")

In [ ]:
import pandas as pd

# Top-K 결과를 CSV로 저장 (분석 및 로깅용)
results_dir     = base_path / 'results'
os.makedirs(results_dir, exist_ok=True)
output_csv_path = results_dir / 'top_k_results.csv'

df_top_k = pd.DataFrame(top_k_results)
df_top_k.to_csv(output_csv_path, index=False)

print(f'Top-K 결과 저장: {output_csv_path}')
print(df_top_k.head())

In [ ]:
# 행렬곱 Retrieval에 사용할 Top-K 문서 인덱스 추출
top_k_indices = [res['id'] for res in top_k_results]
print(f'Top-K 인덱스: {top_k_indices}')

### 4단계: Selector 벡터 생성

Selector 행렬 S (k×n_docs): k번째 행은 k번째 Top-K 문서에 해당하는 one-hot 벡터.  
`S @ EmbDB = (k×n_docs) @ (n_docs×dim) → (k×dim)` 로 Top-K 문서 임베딩을 추출한다.  
Selector를 암호화하면 서버가 어떤 문서가 선택됐는지 알 수 없다.

In [ ]:
doc_count = len(t_encrypted_db_cts)

# 암호화된 selector (프라이버시 보호 행렬곱용)
idx_selector_vector = create_selector_vector(top_k_indices, doc_count)

# 평문 selector (검증 비교용)
idx_selector_vector_pt = create_selector_vector_pt(top_k_indices, doc_count)

### 5단계: 암호화 상태로 Top-K 문서 벡터 복원 (행렬곱)

In [ ]:
# S (k×n_docs, 암호화) × EmbDB_T (n_docs×dim, 열 기준 암호화)
# matmul_res[r][c] = r번째 Top-K 문서의 c번째 차원값 ciphertext
matmul_res = encrypted_matmul(
    idx_selector_vector,   # 암호화된 selector 행렬 (k개 행)
    t_encrypted_db_cts,    # 열 기준 암호화 DB (SLOT_COUNT개 열)
    slot_count,
    engine, rel_key, rot_key
)

In [ ]:
# 행렬곱 결과를 복호화하여 Top-K 문서 임베딩 확인
# res[r][c] = r번째 Top-K 문서의 c번째 차원 스칼라값
res = []
for row_cts in matmul_res:
    row_vals = [engine.decrypt(ct, secret_key)[0] for ct in row_cts]
    res.append(row_vals)

for r_idx, r in enumerate(res):
    print(f'Top-{r_idx+1} 문서 임베딩 (첫 8차원): {r[:8]}')

In [ ]:
# matmul_res shape 확인: (k, dim) — k개 문서 × dim차원
# 각 원소는 ciphertext 객체이므로 np.array 변환은 참조 배열
res_np = np.array(matmul_res, dtype=object)
print(f'matmul_res shape: {res_np.shape}')  # 예상: ({TOP_K_SIZE}, {slot_count})